# ERA5-Land Annual Comparison Sites Run

Run this notebook top to bottom in Google Colab. It launches one Earth Engine table export per year for the 55 comparison sites, using the polygon-first / fine-scale polygon retry workflow.

The comparison site list is embedded in the notebook, so there is no CSV upload step. Colab and Earth Engine may still ask you to authenticate.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess
from pathlib import Path

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run(["pip", "-q", "install", "-r", "requirements.txt"], check=True)


In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")

DRIVE_EXPORT_FOLDER = "watershed_size_gee_exports_snow8day_2001_2023"

assets.setdefault("exports", {}).update(
    {
        "destination": "drive",
        "drive_folder": DRIVE_EXPORT_FOLDER,
    }
)

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

print("Earth Engine project:", assets.get("earth_engine", {}).get("project"))
print("Watershed asset:", assets["watersheds"]["asset_id"])
print("Drive output folder:", assets["exports"].get("drive_folder"))


## Edit Run Settings

Change the settings in the next cell before choosing **Runtime > Run all**. The default is the watershed-size comparison test. You can also switch to all sites or paste a short custom `SITE_IDS` list.


In [ ]:
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

# Site options: "comparison_sites", "all_sites", or "site_id_list".
RUN_SCOPE = "comparison_sites"
RUN_LABEL = f"{RUN_SCOPE}_fine_scale_snow8day"

# Years are inclusive. Use YEARS_TO_SKIP for partial reruns after some exports finish.
START_YEAR = 2001
END_YEAR = 2023
YEARS_TO_SKIP = []

# Leave as None for the full test. Set to 1 or 2 for a quick smoke test.
MAX_TASKS_TO_LAUNCH = None

# Product options: precip, temp, evapotrans, potential_evap, snow_cover, snow_water_equiv.
PRODUCTS = [
    "precip",
    "temp",
    "evapotrans",
    "potential_evap",
    "snow_cover",
    "snow_water_equiv",
]

SITE_IDS = []  # Used only when RUN_SCOPE = "site_id_list".

COMPARISON_SITE_IDS = [
    "and__and_9",
    "and__and_10",
    "and__and_6",
    "and__and_7",
    "and__and_8",
    "and__and_2",
    "and__and_1",
    "and__and_3",
    "and__mack",
    "and__lookoutcreek",
    "catalina_jemez__mg_g_outletq",
    "nwt__martinelli",
    "krycklan__krycklan2",
    "hbr__ws2",
    "mcm__f1_canada_stream_channel",
    "luq__bisley_3",
    "walker_branch__westfork",
    "krycklan__krycklan1",
    "krycklan__krycklan5",
    "hbr__ws7",
    "krycklan__krycklan6",
    "catalina_jemez__marshall_gulch",
    "krycklan__krycklan9",
    "krycklan__krycklan10",
    "nwt__comocreek",
    "nwt__como",
    "usgs__byrn",
    "krycklan__krycklan13",
    "luq__sabana",
    "guadeloupe__petit_bras_david",
    "krycklan__krycklan14",
    "luq__mameyes",
    "eastriversfa__tr_tcg1",
    "usgs__fran",
    "coal_creek__coalcreek",
    "krycklan__krycklan16",
    "usgs__shin",
    "uk__uk_36012",
    "usgs__durh",
    "eastriversfa__ce_cmt0",
    "uk__uk_45009",
    "uk__uk_27030",
    "usgs__prov",
    "usgs__avon",
    "swedish_goverment__swe_gov_gavlean",
    "umr__s000_2k",
    "niva__polygon",
    "usgs__keos",
    "md__md15",
    "hybam__porto_velho",
    "arc__imnavait_upper",
    "arc__imnavait_wt_07_weir",
    "arc__toolik_inlet",
    "arc__tw_weir",
    "coloradoalpine__andrewscreek"
]

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
property_names = watersheds.first().propertyNames().getInfo()
site_id_column = choose_property_name(watersheds, "site_id")
lter_column = choose_property_name(watersheds, "lter", alternatives=["LTER"])

if RUN_SCOPE == "all_sites":
    selected_watersheds = watersheds
elif RUN_SCOPE == "comparison_sites":
    selected_watersheds = watersheds.filter(
        ee.Filter.inList(site_id_column, COMPARISON_SITE_IDS)
    )
elif RUN_SCOPE == "site_id_list":
    if not SITE_IDS:
        raise ValueError('Set SITE_IDS when RUN_SCOPE = "site_id_list".')
    selected_watersheds = watersheds.filter(
        ee.Filter.inList(site_id_column, SITE_IDS)
    )
else:
    raise ValueError('RUN_SCOPE must be "comparison_sites", "all_sites", or "site_id_list".')

selected_row_count = selected_watersheds.size().getInfo()

print("Asset rows:", watersheds.size().getInfo())
print("Asset columns:", property_names)
print("Site ID column:", site_id_column)
print("LTER column:", lter_column)
print("Run scope:", RUN_SCOPE)
print("Run label:", RUN_LABEL)
print("Year range:", f"{START_YEAR}-{END_YEAR}")
print("Years skipped:", YEARS_TO_SKIP)
print("Products:", PRODUCTS)
print("Custom site IDs:", SITE_IDS)
print("Selected watershed rows:", selected_row_count)
print("Selected preview:")
print(selected_watersheds.limit(5).getInfo())

if selected_row_count == 0:
    raise ValueError("No watershed rows were selected.")

expected_site_ids = []
if RUN_SCOPE == "comparison_sites":
    expected_site_ids = COMPARISON_SITE_IDS
elif RUN_SCOPE == "site_id_list":
    expected_site_ids = SITE_IDS

asset_site_ids = set(
    str(value)
    for value in selected_watersheds.aggregate_array(site_id_column).getInfo()
)
missing_site_ids = [
    site_id for site_id in expected_site_ids
    if site_id not in asset_site_ids
]
if missing_site_ids:
    print(
        "Warning: some requested site IDs were not found in the uploaded watershed asset. "
        "Continuing with the asset-present sites."
    )
    print("Missing comparison site IDs:")
    print("\n".join(missing_site_ids))
    print("Selected comparison site IDs found:", len(asset_site_ids))



In [ ]:
from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.extraction import era5_export_columns, extract_era5_land_products
from src.gee_spatial.timing import (
    datetime_label,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

years_to_launch = [
    year for year in range(START_YEAR, END_YEAR + 1)
    if year not in set(YEARS_TO_SKIP)
]
if MAX_TASKS_TO_LAUNCH is not None:
    years_to_launch = years_to_launch[: int(MAX_TASKS_TO_LAUNCH)]

session_started_at = utc_now()
timing_log_path = Path("timing-logs") / (
    f"gee_task_timing_era5_land_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}_"
    f"{datetime_label(session_started_at)}.csv"
)
timing_log_path.parent.mkdir(exist_ok=True)

selectors = era5_export_columns(products, product_names=PRODUCTS, include_snow_comparison_metric=True)
launched_tasks = []

print("Years to launch:", years_to_launch)
print("Years skipped:", YEARS_TO_SKIP)
print("Selected watershed rows:", selected_row_count)
print("Drive output folder:", assets["exports"].get("drive_folder"))
print("Export columns:", selectors)
print("Timing log:", timing_log_path)

for year in years_to_launch:
    out_name = f"era5_land_{year}_{RUN_LABEL}_watershed_extract"
    print()
    print("Launching:", out_name)

    export_rows = extract_era5_land_products(
        products,
        selected_watersheds,
        year=year,
        product_names=PRODUCTS,
        include_snow_comparison_metric=True,
    )

    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )

    launched_at = utc_now()
    run_row = {
        "run_name": f"era5_land_{RUN_LABEL}_annual_{START_YEAR}_{END_YEAR}",
        "mode": "era5_land",
        "products": PRODUCTS,
        "year": year,
        "month": None,
        "months": None,
        "period": "annual",
        "run_group": RUN_LABEL,
    }
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append(
        {
            "name": out_name,
            "task": task,
            "launched_at": launched_at,
            "timing_row": timing_row,
        }
    )
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))

print()
print("Launched tasks:", len(launched_tasks))
print("Timing log:", timing_log_path)


In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60

if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this session.")
else:
    while True:
        now = utc_now()
        print()
        print("Task status at", now.isoformat(timespec="seconds"))
        all_done = True

        for item in launched_tasks:
            item["timing_row"] = update_task_timing_row(
                item["timing_row"],
                item["task"],
                checked_at=now,
            )
            state = item["timing_row"].get("state")
            elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
            print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")

            if state not in DONE_STATES:
                all_done = False

        write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)

        if all_done:
            break

        time.sleep(poll_seconds)

    print_timing_summary(timing_rows_from_launched_tasks(launched_tasks))
    print("Timing log:", timing_log_path)


## After The Exports Finish

This notebook stops after Earth Engine tasks finish and the CSVs are written to the Drive export folder named `watershed_size_gee_exports_snow8day_2001_2023`. Use the local R scripts in `workflow/` to move the CSVs into a tidy run folder and run QA or inventory steps.
